# AI Wine Sommelier RAG

## Wine Review Indexing

https://www.kaggle.com/datasets/christopheiv/winemagdata130k

In [1]:
%pip install -Uqqq langchain langchain-community langchain-openai langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [5]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

## Pinecone 테스트

In [2]:
# 데이터로드
from langchain_core.documents import Document

documents = [
    Document(page_content="LangChain은 LLM 기반 애플리케이션을 쉽게 만들 수 있는 프레임워크입니다.", metadata={"source": "https://langchain.com/docs", "author": "alice", "page": 1}),
    Document(page_content="ChromaDB는 오픈소스 벡터 데이터베이스입니다.", metadata={"source": "https://chromadb.org/intro", "license": "MIT", "date": "2024-07-01"}),
    Document(page_content="파이썬으로 AI 서비스를 개발할 수 있습니다.", metadata={"source": "https://pythonai.co.kr", "editor": "kim", "page": 7}),
    Document(page_content="LLM은 자연어 처리를 위한 대형 언어 모델을 의미합니다.", metadata={"source": "https://llmwiki.com/info", "author": "bob", "version": "v1.1"}),
    Document(page_content="RAG는 검색과 생성의 결합 방식을 제공합니다.", metadata={"source": "https://rag-search.io", "reviewer": "lee", "section": "summary"}),
    Document(page_content="벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.", metadata={"source": "https://vectorbase.net", "author": "jin", "topic": "vector"}),
    Document(page_content="LangChain을 이용하면 다양한 AI 파이프라인을 구축할 수 있습니다.", metadata={"source": "https://langchain.com/blog", "editor": "sarah", "date": "2024-06-30"}),
    Document(page_content="OpenAI의 GPT 모델은 텍스트 생성에 특화되어 있습니다.", metadata={"source": "https://openai.com/gpt", "lang": "ko", "page": 5}),
    Document(page_content="파이썬은 AI 및 데이터 분석 분야에서 널리 사용되는 언어입니다.", metadata={"source": "https://python.org/usecases", "author": "chun", "updated": "2024-05"}),
    Document(page_content="Streamlit은 파이썬으로 대시보드를 쉽게 만들 수 있는 프레임워크입니다.", metadata={"source": "https://streamlit.io/start", "editor": "park", "date": "2024-04-28"}),
    Document(page_content="Dense Retrieval은 임베딩 벡터를 이용한 검색 방식을 의미합니다.", metadata={"source": "https://retrieval.ai/dense", "type": "tech", "page": 3}),
    Document(page_content="Pandas 라이브러리는 데이터 분석에 자주 사용됩니다.", metadata={"source": "https://pandas.pydata.org/about", "maintainer": "koh", "section": "intro"}),
    Document(page_content="메타데이터 필터링은 검색 결과의 품질을 높여줍니다.", metadata={"source": "https://search.com/metadata", "author": "seo", "feature": "filter"}),
    Document(page_content="SelfQueryRetriever는 자연어 쿼리를 임베딩 쿼리로 변환해줍니다.", metadata={"source": "https://selfquery.ai", "editor": "min", "date": "2024-05-12"}),
    Document(page_content="프롬프트 엔지니어링은 LLM의 성능을 극대화하는 방법입니다.", metadata={"source": "https://prompting.dev/guide", "author": "yang", "topic": "prompt"}),
    Document(page_content="HyDE 기법은 하이브리드 검색에 사용됩니다.", metadata={"source": "https://hyde-tech.com", "reviewer": "kang", "version": "2024.1"}),
    Document(page_content="CoT는 복잡한 문제를 단계적으로 해결하는 프롬프트 기법입니다.", metadata={"source": "https://cotprompt.org", "editor": "jung", "date": "2023-12-01"}),
    Document(page_content="문서 임베딩은 텍스트를 고차원 벡터로 변환하는 과정입니다.", metadata={"source": "https://embedding.ai/intro", "section": "embedding", "author": "song"}),
    Document(page_content="CrewAI는 멀티 에이전트 시스템 구현을 돕는 툴입니다.", metadata={"source": "https://crew.ai/docs", "lang": "ko", "page": 9}),
    Document(page_content="Fine-tuning은 사전학습 모델을 특정 도메인에 맞게 재학습시키는 과정입니다.", metadata={"source": "https://finetune.ai/guide", "editor": "jeon", "date": "2024-01-30"})
]

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model='text-embedding-3-small') # 1536차원 임베딩 모델
# 문서 -> 임베딩 -> Pinecone 업로드
vector_store = PineconeVectorStore.from_documents(
    documents, # list[Document]
    embeddings,
    index_name = 'pinecone-test' # 저장할 index명
)

In [7]:
# 코사인 유사도로 검색
retriever = vector_store.similarity_search('벡터 데이터베이스란?')
retriever

[Document(id='9766dc99-d428-4b50-9e01-8b697b1c24ca', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.'),
 Document(id='e95abfa0-1425-4091-b497-c4871b82b61b', metadata={'date': '2024-07-01', 'license': 'MIT', 'source': 'https://chromadb.org/intro'}, page_content='ChromaDB는 오픈소스 벡터 데이터베이스입니다.'),
 Document(id='f63719e4-706e-489f-a76a-c4dad49070c0', metadata={'page': 3.0, 'source': 'https://retrieval.ai/dense', 'type': 'tech'}, page_content='Dense Retrieval은 임베딩 벡터를 이용한 검색 방식을 의미합니다.'),
 Document(id='79541ca2-e034-4ab7-be3d-ebe8d3168841', metadata={'maintainer': 'koh', 'section': 'intro', 'source': 'https://pandas.pydata.org/about'}, page_content='Pandas 라이브러리는 데이터 분석에 자주 사용됩니다.')]

In [8]:
# 벡터스토어를 Retriever 인터페이스로 변환
retriever = vector_store.as_retriever(
    search_typ = 'similarity',
    search_kwargs = {'k': 5}
)
retriever.invoke('벡터 데이터베이스란?')

[Document(id='9766dc99-d428-4b50-9e01-8b697b1c24ca', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.'),
 Document(id='e95abfa0-1425-4091-b497-c4871b82b61b', metadata={'date': '2024-07-01', 'license': 'MIT', 'source': 'https://chromadb.org/intro'}, page_content='ChromaDB는 오픈소스 벡터 데이터베이스입니다.'),
 Document(id='f63719e4-706e-489f-a76a-c4dad49070c0', metadata={'page': 3.0, 'source': 'https://retrieval.ai/dense', 'type': 'tech'}, page_content='Dense Retrieval은 임베딩 벡터를 이용한 검색 방식을 의미합니다.'),
 Document(id='79541ca2-e034-4ab7-be3d-ebe8d3168841', metadata={'maintainer': 'koh', 'section': 'intro', 'source': 'https://pandas.pydata.org/about'}, page_content='Pandas 라이브러리는 데이터 분석에 자주 사용됩니다.'),
 Document(id='77ee6bdf-2aaf-41e5-bf84-7e67ceec13fd', metadata={'author': 'song', 'section': 'embedding', 'source': 'https://embedding.ai/intro'}, page_content='문서 임베딩은 텍스트를 고차원 벡터로 변환하는 과정입니다.')]

In [9]:
!gdown 1sJzwWez1Q7hZqlWOV70Ng00_taDqYPi3

Downloading...
From: https://drive.google.com/uc?id=1sJzwWez1Q7hZqlWOV70Ng00_taDqYPi3
To: c:\Workspace\LLM\06_2stage-rag\winemag-data-130k-v2.csv

  0%|          | 0.00/52.9M [00:00<?, ?B/s]
  1%|          | 524k/52.9M [00:00<00:46, 1.14MB/s]
  2%|▏         | 1.05M/52.9M [00:00<00:26, 1.92MB/s]
  4%|▍         | 2.10M/52.9M [00:00<00:14, 3.60MB/s]
  7%|▋         | 3.67M/52.9M [00:00<00:07, 6.21MB/s]
 11%|█         | 5.77M/52.9M [00:00<00:04, 9.65MB/s]
 17%|█▋        | 8.91M/52.9M [00:01<00:03, 14.5MB/s]
 23%|██▎       | 12.1M/52.9M [00:01<00:02, 17.9MB/s]
 31%|███       | 16.3M/52.9M [00:01<00:01, 23.2MB/s]
 38%|███▊      | 19.9M/52.9M [00:01<00:01, 25.3MB/s]
 45%|████▍     | 23.6M/52.9M [00:01<00:01, 27.2MB/s]
 51%|█████     | 26.7M/52.9M [00:01<00:00, 27.1MB/s]
 56%|█████▋    | 29.9M/52.9M [00:01<00:00, 27.3MB/s]
 62%|██████▏   | 33.0M/52.9M [00:01<00:00, 27.5MB/s]
 68%|██████▊   | 36.2M/52.9M [00:02<00:00, 28.3MB/s]
 74%|███████▍  | 39.3M/52.9M [00:02<00:00, 25.9MB/s]
 82%|████████▏ 

In [ ]:
# CSVLoader : CSV의 각 행을 하나의 Document로 변환하는 로더
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader('winemag-data-130k-v2.csv', encoding='utf-8')
docs = loader.load() # CSV를 -> list[Document]
print(len(docs))


129971


In [ ]:
for i, doc in enumerate(docs[:2]):
    print(f"{i} : {type(doc)}")
    print(doc.metadata)
    print(doc.page_content)
    print()


0 : <class 'langchain_core.documents.base.Document'>
{'source': 'winemag-data-130k-v2.csv', 'row': 0}
: 0
country: Italy
description: Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.
designation: Vulkà Bianco
points: 87
price: 
province: Sicily & Sardinia
region_1: Etna
region_2: 
taster_name: Kerin O’Keefe
taster_twitter_handle: @kerinokeefe
title: Nicosia 2013 Vulkà Bianco  (Etna)
variety: White Blend
winery: Nicosia

1 : <class 'langchain_core.documents.base.Document'>
{'source': 'winemag-data-130k-v2.csv', 'row': 1}
: 1
country: Portugal
description: This is ripe and fruity, a wine that is smooth while still structured. Firm tannins are filled out with juicy red berry fruits and freshened with acidity. It's  already drinkable, although it will certainly be better from 2016.
designation: Avidagos
points: 87
price: 15.0
province: Douro
region_1: 
region_2: 
tast

In [16]:
# Pinecone 업로드 : Pinecone 인덱스에 연결된 벡터스토어 객체 생성
vector_store = PineconeVectorStore(
    index_name = 'winemag-data',
    embedding = embeddings
)

batch_size = 100

for i in range(0, len(docs), batch_size): # batch_size 단위 순회
    batch_data = docs[i:i+batch_size] # batch_size 만큼씩 가져옴
    vector_store.add_documents(batch_data)
    print(f"index: {i} ~ {i + batch_size}")

index: 0 ~ 100
index: 100 ~ 200
index: 200 ~ 300
index: 300 ~ 400
index: 400 ~ 500
index: 500 ~ 600
index: 600 ~ 700
index: 700 ~ 800
index: 800 ~ 900
index: 900 ~ 1000
index: 1000 ~ 1100
index: 1100 ~ 1200
index: 1200 ~ 1300
index: 1300 ~ 1400
index: 1400 ~ 1500
index: 1500 ~ 1600
index: 1600 ~ 1700
index: 1700 ~ 1800
index: 1800 ~ 1900
index: 1900 ~ 2000
index: 2000 ~ 2100
index: 2100 ~ 2200
index: 2200 ~ 2300
index: 2300 ~ 2400
index: 2400 ~ 2500
index: 2500 ~ 2600
index: 2600 ~ 2700
index: 2700 ~ 2800
index: 2800 ~ 2900
index: 2900 ~ 3000
index: 3000 ~ 3100
index: 3100 ~ 3200
index: 3200 ~ 3300
index: 3300 ~ 3400
index: 3400 ~ 3500
index: 3500 ~ 3600
index: 3600 ~ 3700
index: 3700 ~ 3800
index: 3800 ~ 3900
index: 3900 ~ 4000
index: 4000 ~ 4100
index: 4100 ~ 4200
index: 4200 ~ 4300
index: 4300 ~ 4400
index: 4400 ~ 4500
index: 4500 ~ 4600
index: 4600 ~ 4700
index: 4700 ~ 4800
index: 4800 ~ 4900
index: 4900 ~ 5000
index: 5000 ~ 5100
index: 5100 ~ 5200
index: 5200 ~ 5300
index: 5300 ~ 

KeyboardInterrupt: 

## Retrieval & Generation
1. 텍스트/이미지 입력으로 요리에 설명 chain
2. 요리설명텍스트 벡터db조회 chain
3. 요리설명/리뷰검색을 가지고 와인추천 응답 chain

### 요리설명 chain

In [ ]:
# 채팅 프롬프트 / 휴먼 메시지 템플릿
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

def describe_dish_flavor(query: dict):
    prompt = ChatPromptTemplate.from_messages([
        ('system', '''
**페르소나 (Persona):**
당신은 식재료의 분자 단위까지 이해하는 '미식의 철학자'이자, 절대미각을 지닌 최고 수준의 푸드 칼럼니스트이다.
당신은 요리를 단순한 음식이 아닌, 식재료와 조리 과학(Culinary Science)이 빚어낸 예술 작품으로 바라본다.
당신의 표현은 식재료의 기원부터 조리 과정에서 일어나는 화학적 변화(마이야르 반응, 캐러멜라이징 등)를 아우르며, 읽는 이가 마치 그 음식을 입안에 넣은 듯한 착각을 불러일으킬 정도로 정교하고 관능적이다.

**역할 (Role):**
당신의 핵심 역할은 요리의 맛, 향, 텍스처(Texture), 그리고 밸런스를 해부학적으로 분석하여 전달하는 것이다.
1.  **다차원적 분석:** 맛을 평면적으로 묘사하지 않고, '첫맛(Attack) - 중간 맛(Mid-palate) - 끝맛(Finish)'의 시퀀스로 나누어 입체적으로 설명한다.
2.  **조리법과 맛의 인과관계:** 왜 이 맛이 나는지, 어떤 조리 테크닉이 식재료의 잠재력을 폭발시켰는지 논리적 근거를 제시한다.
3.  **미식의 가이드:** 식재료 간의 궁합(Pairing)과 풍미를 극대화하는 팁을 제공하여, 사용자의 미식 수준을 한 단계 끌어올린다.

**가이드라인 (Guidelines):**
- **감각의 구체화:** '맛있다', '부드럽다' 같은 추상적 표현을 금지한다. 대신 '혀를 감싸는 벨벳 같은 질감', '비강을 때리는 훈연 향' 등 구체적인 묘사를 사용하라.
- **단계별 서술:** 시각과 후각으로 시작해, 입안에서의 질감 변화, 그리고 목 넘김 후의 여운까지 단계별로 서술하라.

**예시 (Examples):**

* **사용자:** "잘 만든 '트러플 크림 리조또'의 맛을 묘사해 주세요."
    **당신:**
    * **[시각과 후각]** 김이 모락모락 나는 접시 위로 흙내음(Earthy)을 가득 머금은 트러플 향이 가장 먼저 코끝을 강타합니다. 크림소스의 녹진한 유분 향과 섞여 마치 가을 숲속에 와 있는 듯한 묵직한 아로마가 식욕을 자극합니다.
    * **[첫맛과 텍스처]** 한 숟가락 입에 넣으면, 알덴테(Al dente)로 익혀 심지가 살아있는 쌀알이 혀 위에서 경쾌하게 굴러다닙니다. 동시에 파르미지아노 레지아노 치즈가 녹아든 크림소스가 쌀알 사이사이를 끈적하게 메우며 혀를 포근하게 감싸 안습니다.
    * **[풍미의 폭발]** 씹을수록 버섯의 감칠맛(Umami)이 폭발합니다. 버터의 고소함이 베이스를 깔아주는 가운데, 트러플 오일의 강렬한 향이 비강으로 역류하며 미각을 지배합니다.
    * **[여운]** 목을 넘긴 후에도 트러플의 진한 향과 크림의 고소함이 입안에 길게 남아, 무거운 레드 와인 한 모금을 간절하게 부릅니다.

* **사용자:** "양파 수프(French Onion Soup)의 맛의 비결이 무엇인가요?"
    **당신:**
    * **[핵심 분석]** 이 요리의 영혼은 **'인내심이 만든 단맛'**에 있습니다. 양파를 약불에서 장시간 볶아내는 '캐러멜라이징(Caramelization)' 과정이 핵심입니다.
    * **[맛의 레이어]** 양파의 매운 성분이 열을 만나 짙은 갈색의 끈적한 당분으로 변하며, 설탕과는 차원이 다른 깊고 복합적인 단맛을 냅니다. 여기에 쇠고기 육수의 짭조름한 감칠맛이 더해져 '단짠'의 완벽한 균형을 이룹니다.
    * **[식감의 조화]** 흐물흐물하게 녹아내린 양파와 국물을 머금어 축축해진 바게트, 그리고 그 위를 덮은 그뤼에르 치즈의 쫄깃함이 섞이며 입안 가득 풍성한 식감의 축제를 엽니다.

**주의사항**
맛의 대한 묘사만 줄글 형식으로 50자이내로 작성하세요.
'''),
        ('human', '사용자가 제공한 이미지의 요리명과 풍미를 잘 묘사해 주세요. ')
    ])

    temp = []
    # image_urls가 있는 경우 이미지 URL들을 메시지 블록으로 추가
    if query.get('image_urls'):
        temp += [{'image_url' : image_url} for image_url in query.get('image_urls')]
    # text가 있는 경우 메시지 블록으로 추가
    if query.get('text'):
        temp += [{'text' : query.get('text')}]
    # HumanMessagePromptTemplate : 멀티모달 블록형태의 값을 human 메시지로 프롬프트에 추가
    prompt += HumanMessagePromptTemplate.from_template(temp)

    llm = init_chat_model('gpt-5.6-luna')
    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    return chain # 체인 결과 반환

# 입력을 받아 chain을 실행할 수 있는 Runnable
dish_flavor_chain = RunnableLambda(describe_dish_flavor)
response = dish_flavor_chain.invoke({
    'text': '',
    'image_urls': [
        'https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMjVfODIg%2FMDAxNzYxMzc2NjMzMDA4.-qxYpSDZfPleD8cj9VzxvqckYRIvaGpZW-fibT3whjsg.mefkB_k7NzVsXrb9RGPEQvZAplyzrustInLMV-827Gkg.JPEG%2FIMG%25A3%25DF2865.JPG&type=sc960_832',
        'https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTA5MjBfMjEz%2FMDAxNzU4MzgwMDM4MzAz.XWaCn8Xu_7jjbYWq5P4MFqibaqNz4p3CFKRgjOnP2dMg.7wrfjF9U-p-CORf9ix4DbEGFRnOkaNh2ihjYlZOZy6Ag.JPEG%2FIMG_6083.JPG&type=sc960_832'
    ]
})

print(response)

바삭한 돈가스와 매콤달콤한 제육볶음, 고소한 육즙이 어우러진 풍성한 한상입니다.


## 리뷰 검색 chain

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 요리 풍미 설명을 받아서 Pinecone에서 유사한 와인리뷰를 찾아 반환하는 함수
def search_wine_review(query):
    embeddings = OpenAIEmbeddings(model='text-embedding-3-small') # 1536차원 임베딩 모델
    # PineconeVectorStore index 연결
    vector_store = PineconeVectorStore(
        index_name = 'winemag-data', # 검색할 index명
        embedding=embeddings,
    )
    docs = vector_store.similarity_search(query, k=5)

    return {
        'dish_flavor': query,
        'wine_reviews': '\n\n'.join(doc.page_content for doc in docs)
    }

query = '''
첫 번째 이미지는 '허브 그릴 스테이크'입니다. 입안에서 육즙과 허브 오일이 조화를 이루며 풍부하고 깊은 감칠맛이 혀를 감싸고, 구운 토마토의 은은한 산미가 중간 맛에 신선함을 부여합니다.
두 번째 이미지는 '시저 샐러드'입니다. 크리스피한 크루통과 신선한 로메인 상추가 바삭한 질감을 선사하고, 고소한 파마산 치즈와 크리미한 시저 드레싱이 입안 가득 고소함과 산뜻한 여운을 남깁니다.
'''

result = search_wine_review(query)

print(result['wine_reviews'])

: 4988
country: US
description: Honey-sweet and direct, with citrus jam, apricot essence, crème brûlée and vanilla cream flavors. Fine with white cookies, lemon chiffon pie, pineapple sorbet.
designation: Madeline
points: 88
price: 35.0
province: California
region_1: Napa Valley
region_2: Napa
taster_name: 
taster_twitter_handle: 
title: Prager 2004 Madeline Riesling (Napa Valley)
variety: Riesling
winery: Prager

: 8158
country: US
description: A sweet smell of honeysuckle and grapefruit candy permeates the nose of this bottling, along with cut honeydew melon and apple blossom. Sugary mandarin juice is the primary flavor on the palate, but it's properly offset by acidity a chalky texture.
designation: 
points: 87
price: 29.0
province: California
region_1: Central Coast
region_2: 
taster_name: Matt Kettmann
taster_twitter_handle: @mattkettmann
title: Coquelicot 2016 Riesling
variety: Riesling
winery: Coquelicot

: 12195
country: US
description: Soft and melted in texture, with flavors 

In [ ]:
# 파이프라인 중간점검 (요리 풍미 추출 -> 와인 리뷰 검색)

search_wine_review_chain = RunnableLambda(search_wine_review) # 함수 -> Runnable
# 이미지,텍스트 -> 풍미 | 풍미 -> 리뷰 검색(벡터DB)
chain = dish_flavor_chain | search_wine_review_chain

response = chain.invoke({
    'text': '',
    'image_urls': [
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMTNfMjM4%2FMDAxNzYwMzYzODc4NDk0.me3_kw-eBIiNd75S0W7XdkBiVbUn78pRz5cVlQWV0EEg.VQcftg1p19qRbhxcvaT9Rk80wha9g1VD0f5yVaIe7cUg.PNG%2FImage_fx_%252866%2529.png&type=sc960_832",
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAxODExMjhfMjEy%2FMDAxNTQzNDE1MzEzOTQw.8BT4PzdMbbRctmFFAkLWz3p3G0KywePJxA70TlwTQjcg.8FTmR-SX6Wt7ey27RbV78uzAD8AqedJJKbaAdduuI3Qg.JPEG.story77616%2F20181128_121945.jpg&type=sc960_832"
    ]
})
print(response)

{'dish_flavor': '허브 스테이크는 육즙과 훈연향, 시저 샐러드는 고소하고 산뜻한 풍미.', 'wine_reviews': ": 26833\ncountry: US\ndescription: The trick with sparkling wine is to achieve finesse. This Pinot Noir-Chardonnay blend is too scoury in bubbles, giving it a rough feel. Nonetheless it's delicious and easy to like for its yeasty flavors of limes, oranges and vanilla honey.\ndesignation: Brut\npoints: 87\nprice: 45.0\nprovince: California\nregion_1: Sta. Rita Hills\nregion_2: Central Coast\ntaster_name: \ntaster_twitter_handle: \ntitle: Kessler-Haak 2012 Brut Sparkling (Sta. Rita Hills)\nvariety: Sparkling Blend\nwinery: Kessler-Haak\n\n: 5664\ncountry: US\ndescription: The winery's annual stainless-steel bottling, this is showy rather than reserved, with ripe Asian pear, tropical flowers, sweet apple blossoms, pineapple and a bit of vanilla on the nose. Nicely fresh, it's bright on the palate, with a sizzle of acidity and grippy chalkiness that frame the poached-pear palate.\ndesignation: Steel\npoints: 90\npri

In [23]:
response = chain.invoke({'text': '오늘 저녁은 버터와 허브로 구운 캐비어 가리비 관자 구이를 먹겠다.'})

In [24]:
print(response)

{'dish_flavor': '버터 허브 가리비 관자 구이: 달큰한 육질에 캐비어의 짭조름한 감칠맛이 퍼진다.', 'wine_reviews': ": 4012\ncountry: South Africa\ndescription: There's a lush, decadent edge to this wine, thanks to heady aromas of vanilla cream, chocolate-covered raspberry and boysenberry preserves. The palate shows more restraint, with textured, fruit-skin flavors of plum and cherry that finish slightly chewy. Enjoy this now.\ndesignation: The Ambassador\npoints: 88\nprice: 25.0\nprovince: Stellenbosch\nregion_1: \nregion_2: \ntaster_name: Lauren Buzzeo\ntaster_twitter_handle: @laurbuzz\ntitle: Beau Joubert 2009 The Ambassador Red (Stellenbosch)\nvariety: Red Blend\nwinery: Beau Joubert\n\n: 12067\ncountry: US\ndescription: Overtly sweet, this tastes like a dessert pastry turned into wine. The flavors are of orange, pineapple, vanilla-cream and buttered toast.\ndesignation: Back Seat Blonde\npoints: 82\nprice: 14.0\nprovince: California\nregion_1: Santa Ynez Valley\nregion_2: Central Coast\ntaster_name: \ntaster_twitter

In [27]:
# 와인 추천 체인 : 요리 풍미 + 검색된 와인 리뷰를 바탕으로 페어링 추천
def recommend_wines(query):
    prompt = ChatPromptTemplate.from_messages([
        ('system', '''
**페르소나 (Persona):**
당신은 와인과 미식의 조화로운 세계를 탐험하는 '마리아주(Mariage)의 설계자'이자 경험 풍부한 소믈리에이다.
당신은 전 세계의 와인 산지와 품종에 대한 백과사전적 지식을 갖추고 있으며, 복잡한 와인 용어를 누구나 이해하기 쉬운 감각적인 언어로 풀어내는 탁월한 능력을 지녔다.
당신의 태도는 언제나 환대하는 마음(Hospitality)으로 가득 차 있어, 와인 초보자부터 애호가까지 모두를 편안하게 이끈다.

**역할 (Role):**
당신의 유일하고도 가장 중요한 역할은 사용자가 준비한 요리에 **'영혼의 단짝'이 될 와인을 추천**하는 것이다.
1.  **미각 분석:** 요리의 주재료, 소스, 조리법(굽기, 찌기 등)을 분석하여 맛의 무게감과 특성을 파악한다.
2.  **정밀한 페어링:** 산도(Acidity), 당도(Sweetness), 타닌(Tannin), 바디감(Body)의 균형을 고려해 와인을 선정한다.
3.  **이유 설명:** 단순히 와인 이름만 던지는 것이 아니라, **"왜 이 와인이 그 음식과 어울리는지"** 미각적, 화학적 근거를 들어 설득력 있게 설명한다.

**가이드라인 (Guidelines):**
- **음식 중심 예시:** 모든 답변은 구체적인 요리에 대한 와인 추천으로 이루어져야 한다.
- **상호보완의 원리:** 와인이 음식의 맛을 어떻게 상승시키는지(증폭), 혹은 음식의 단점을 어떻게 가려주는지(보완) 묘사하라.

**예시 (Examples):**
... (생략) ...
'''),
        # 입력 변수(dish_flavor, wine_reviews) 기반 요청
        ('human', '''
와인페이링 추천에 있어 아래 제시된 요리와 풍미, 와인리뷰만을 기초하여 답변해주세요.

## 요리와 풍미 ##
{dish_flavor}

## 와인리뷰 정보 ##
{wine_reviews}
''')
    ])

    llm = init_chat_model('gpt-5.6-luna')
    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    return chain # 체인 결과 반환

recommend_wines_chain = RunnableLambda(recommend_wines)
response = recommend_wines_chain.invoke({
    'dish_flavor': '허브 스테이크는 육즙과 훈연향, 시저 샐러드는 고소하고 산뜻한 풍미.', 'wine_reviews': ": 26833\ncountry: US\ndescription: The trick with sparkling wine is to achieve finesse. This Pinot Noir-Chardonnay blend is too scoury in bubbles, giving it a rough feel. Nonetheless it's delicious and easy to like for its yeasty flavors of limes, oranges and vanilla honey.\ndesignation: Brut\npoints: 87\nprice: 45.0\nprovince: California\nregion_1: Sta. Rita Hills\nregion_2: Central Coast\ntaster_name: \ntaster_twitter_handle: \ntitle: Kessler-Haak 2012 Brut Sparkling (Sta. Rita Hills)\nvariety: Sparkling Blend\nwinery: Kessler-Haak\n\n: 5664\ncountry: US\ndescription: The winery's annual stainless-steel bottling, this is showy rather than reserved, with ripe Asian pear, tropical flowers, sweet apple blossoms, pineapple and a bit of vanilla on the nose. Nicely fresh, it's bright on the palate, with a sizzle of acidity and grippy chalkiness that frame the poached-pear palate.\ndesignation: Steel\npoints: 90\nprice: 30.0\nprovince: California\nregion_1: Sta. Rita Hills\nregion_2: Central Coast\ntaster_name: Matt Kettmann\ntaster_twitter_handle: @mattkettmann\ntitle: Foley 2014 Steel Chardonnay (Sta. Rita Hills)\nvariety: Chardonnay\nwinery: Foley\n\n: 32737\ncountry: US\ndescription: This esteemed winery's annual bubbly is crafted much like its still Chardonnays, showing focused aromas of chalk, lemon zest and a telltale brie cheese rind dairy element. It's very yeasty and slightly sour on the mouthwateringly sharp palate, with squeezed limes, lemon pith and underripe kumquat flavors.\ndesignation: 3-D Sparkling\npoints: 90\nprice: 68.0\nprovince: California\nregion_1: Sta. Rita Hills\nregion_2: Central Coast\ntaster_name: Matt Kettmann\ntaster_twitter_handle: @mattkettmann\ntitle: Brewer-Clifton 2012 3-D Sparkling Chardonnay (Sta. Rita Hills)\nvariety: Chardonnay\nwinery: Brewer-Clifton\n\n: 29012\ncountry: US\ndescription: Shows many of the qualities of Schramsberg's more expensive sparklers, except the bubbles aren't quite as refined. The flavors are rich and satisfying in strawberries, raspberries, vanilla and toast.\ndesignation: Mirabelle Brut\npoints: 87\nprice: 23.0\nprovince: California\nregion_1: North Coast\nregion_2: North Coast\ntaster_name: \ntaster_twitter_handle: \ntitle: Schramsberg NV Mirabelle Brut Sparkling (North Coast)\nvariety: Sparkling Blend\nwinery: Schramsberg\n\n: 25145\ncountry: US\ndescription: A pink-tinged copper color and a pleasing blend of flavors make this sparkling wine from Schramsberg easy to enjoy. The aromas suggest cherries and cinnamon, the flavor is like tart raspberry and the texture is smooth with fine bubbles.\ndesignation: Brut Rose\npoints: 88\nprice: 28.0\nprovince: California\nregion_1: California\nregion_2: California Other\ntaster_name: Jim Gordon\ntaster_twitter_handle: @gordone_cellars\ntitle: Mirabelle NV Brut Rose Sparkling (California)\nvariety: Sparkling Blend\nwinery: Mirabelle"
    })

print(response)

## 가장 추천: **Schramsberg NV Mirabelle Brut Sparkling**

허브 스테이크의 **육즙과 훈연향**, 시저 샐러드의 **고소하고 산뜻한 풍미**를 한 번에 연결하기에 가장 적합합니다.

- **딸기·라즈베리 풍미**: 스테이크의 짭짤하고 구수한 맛에 산뜻한 과실감을 더해, 고기의 풍미가 무겁게 남지 않게 합니다.
- **바닐라와 토스트 향**: 스테이크의 훈연향과 자연스럽게 이어지는 구수한 뉘앙스를 만듭니다.
- **풍부하고 만족스러운 맛**: 다른 스파클링보다 존재감이 있어 허브 스테이크에 묻히지 않습니다.
- **기포**: 시저 샐러드의 고소하고 크리미한 느낌 뒤를 깔끔하게 정리해 다음 한입을 산뜻하게 만들어 줍니다.

### 페어링의 핵심
이 와인은 스테이크의 훈연향을 **토스트와 바닐라 풍미로 증폭**하고, 샐러드의 고소함은 **기포와 과실감으로 보완**합니다. 제공된 와인 중에서는 두 요리 모두와 가장 균형 있게 어울리는 선택입니다.

### 대안
- **Mirabelle NV Brut Rose**: 스테이크보다 시저 샐러드 쪽의 산뜻함을 더 강조하고 싶을 때 좋습니다. 체리와 시나몬, 타르트 라즈베리, 부드럽고 섬세한 기포가 특징입니다.
- **Foley 2014 Steel Chardonnay**: 샐러드 중심의 페어링이라면 적합합니다. 배와 파인애플의 풍미, 생생한 산도, 초크 같은 질감이 고소한 샐러드를 산뜻하게 받쳐주지만, 스테이크의 훈연향에는 미라벨 브뤼가 더 잘 대응합니다.


### 통합 chain

In [ ]:
# {'text': ... , 'image_urls': ...} -> 요리 풍미(텍스트)
dish_flavor_chain = RunnableLambda(describe_dish_flavor)
# 풍미 텍스트 -> 유사한 와인 리뷰 검색 -> {'dish_flavor': ... , 'wine_reviews': ...}
search_wine_review_chain = RunnableLambda(search_wine_review)
# {'dish_flavor': ... , 'wine_reviews': ...} -> 최종 와인 페어링 추천
recommend_wines_chain = RunnableLambda(recommend_wines)


chain = dish_flavor_chain | search_wine_review_chain | recommend_wines_chain

response = chain.invoke({
    'text': '',
    'image_urls': [
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMTNfMjM4%2FMDAxNzYwMzYzODc4NDk0.me3_kw-eBIiNd75S0W7XdkBiVbUn78pRz5cVlQWV0EEg.VQcftg1p19qRbhxcvaT9Rk80wha9g1VD0f5yVaIe7cUg.PNG%2FImage_fx_%252866%2529.png&type=sc960_832",
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAxODExMjhfMjEy%2FMDAxNTQzNDE1MzEzOTQw.8BT4PzdMbbRctmFFAkLWz3p3G0KywePJxA70TlwTQjcg.8FTmR-SX6Wt7ey27RbV78uzAD8AqedJJKbaAdduuI3Qg.JPEG.story77616%2F20181128_121945.jpg&type=sc960_832"
    ]
})

print(response)

## 추천: **Mirabelle NV Brut Rosé Sparkling (California)**
- **평점:** 88점  
- **가격:** $28  
- **풍미:** 체리, 시나몬, 타르트 라즈베리  
- **질감:** 부드럽고 고운 기포  

허브 스테이크의 **육즙과 깊은 풍미**에는 이 와인의 체리와 라즈베리 계열 과실 풍미가 잘 어울립니다. 붉은 과실의 상큼함이 스테이크의 진한 맛을 압도하지 않고, 고기 풍미를 산뜻하게 들어 올려줍니다. 특히 로제의 은은한 과실감은 허브의 향과도 자연스럽게 연결됩니다.

시저샐러드의 **고소한 맛과 아삭한 식감**에는 스파클링의 기포와 산미가 효과적입니다. 기포가 고소한 풍미가 남기는 묵직함을 씻어내고, 다음 한입을 더욱 산뜻하게 만들어줍니다. 부드럽고 고운 기포라서 샐러드의 아삭한 질감도 해치지 않습니다.

### 서빙 팁
차갑게, **약 7~9°C**로 제공하면 좋습니다. 스테이크만 놓고 보면 더 묵직한 레드 와인이 자연스럽지만, 제시된 와인 중에서는 이 로제 스파클링이 **스테이크의 풍미와 시저샐러드의 고소함을 동시에 보완하는 가장 균형 잡힌 선택**입니다.


In [29]:
response = chain.invoke({
    'text': '이따 피자스쿨 베이컨포테이토 피자 먹을거다. 와인은 뭘 먹으면 좋을까?',
    'image_urls': [
        'https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyMTExMjZfOTAg%2FMDAxNjM3ODc3MTI3MDY4.GrivVlwn1aHP8ydQZIA7mym_2AUzq7UaB8zqoTc1d8Mg.z6fe2yK3Tq9E95fHOpHJBq-c3fRUUTGPemiSY939QBcg.JPEG.kkuljo%2F20200406_210410.jpg&type=sc960_832'
    ]
})

print(response)

## 추천: Dr. H. Thanisch 2014 Bernkasteler Badstube Kabinett Riesling

베이컨의 **훈연 향과 짠맛**, 감자의 **고소하고 담백한 풍미**에는 제시된 와인 중 이 모젤 리슬링이 가장 잘 맞습니다.

- **산뜻한 산도**가 베이컨의 짠맛과 기름진 느낌을 깔끔하게 씻어냅니다.
- 사과·배·핵과류의 **신선한 과실 풍미**가 감자의 담백하고 고소한 맛에 밝은 인상을 더합니다.
- 카비넷의 **섬세한 단맛**은 훈연 베이컨의 짭짤함을 부드럽게 감싸며, 음식의 풍미를 더욱 선명하게 해줍니다.
- 긴 여운과 강철 같은 미네랄감은 감자의 묵직함을 가볍게 정리해줍니다.

요리에 산뜻한 **스파클링 와인**이 잘 맞는다는 점을 고려하면, 이 와인은 스파클링은 아니지만 제시된 목록 중에서는 가장 가까운 선택입니다. 특히 충분히 차갑게 마시면 더욱 상쾌하게 즐길 수 있습니다.

### 차선의 선택
**Binz 2013 Nackenheimer Kabinett Trocken Pinot Gris**도 상쾌하고 톡 쏘는 산미와 미네랄 풍미가 있어 괜찮지만, 드라이하고 다소 날렵한 스타일이라 베이컨의 짠맛을 감싸는 힘은 리슬링보다 약합니다.

반면 카베르네 소비뇽과 피노 누아는 잼·오크·버터스카치 같은 풍부한 풍미와 구조감이 있어, 이 담백한 감자와 훈연 베이컨 조합에는 다소 무겁습니다.


In [30]:
response = chain.invoke({
    'text': '고추잡채',
    'image_urls': [
        'https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMDNfNDYg%2FMDAxNzU5NDkwODcxMjE4.nDH5zOdTi5HvhLUtgYXtl9ps1jQLWn5MNC1XNrChF8Yg.xADP7ofugONTY3vwbnFeS7cQQmPTBXQfjC87WNsFF3Ig.JPEG%2FIMG_3801.JPG&type=sc960_832'
    ]
})

print(response)

## 가장 추천하는 와인  
### **Kayra 2012 Öküzgözü (Elazığ) — 90점, $29**

이 요리에는 **터키의 오쿄즈괴주(Öküzgözü)**가 가장 잘 어울립니다.

- **윤기 흐르는 돼지고기**의 풍부함을 와인의 적당한 무게감이 받쳐줍니다.  
- 리뷰에서 언급된 **“pervasive freshness and brightness”**는 돼지고기의 기름진 감촉을 씻어내며 다음 한입을 가볍게 만들어줍니다.  
- **부드러운 타닌**은 돼지고기의 감칠맛과 충돌하지 않고 매끄럽게 이어집니다.  
- 와인의 **블랙 페퍼와 아니제트 향**은 삼색 피망의 알싸한 향과 자연스럽게 호응합니다.  
- 라즈베리와 크랜베리의 산뜻한 붉은 과실 풍미는 피망의 아삭하고 생생한 인상을 살리며, 긴 여운은 요리의 감칠맛과 조화를 이룹니다.

즉, 이 와인은 돼지고기의 풍부함은 받아주고, 피망의 알싸함과 식감은 산뜻하게 끌어올리는 **균형형 페어링**입니다.

## 대안 추천

### 2순위: **Thymiopoulos 2010 Young Vines Xinomavro (Naoussa) — 85점, $18**
훈연 소시지, 후추, 정향, 가죽 향과 함께 **크고 스파이시하며 씹히는 질감**을 지닌 와인입니다.  
돼지고기와 향신료 풍미에는 잘 맞지만, 요리의 섬세한 피망 향보다는 와인의 스모키하고 강한 인상이 앞설 수 있습니다. 고기를 진하게 볶았거나 불향이 더해진 경우에 특히 적합합니다.

### 3순위: **Wrath 2010 Ex Anima Syrah (Monterey) — 88점, $25**
강렬한 블랙베리와 체리, 폭발적인 블랙 페퍼 풍미가 있어 돼지고기와 피망의 조합에 연결점이 있습니다. 다만 리뷰에서도 **짠 고기가 필요하다**고 할 만큼 힘이 강하므로, 요리가 충분히 짭짤하고 진할 때 선택하는 편이 좋습니다.

## 덜 추천하는 와인

- **Oveja Negra Chardonnay-Viognier**: 산도가 거의 없고 무겁고 달게 느